# Load data and cleaning

In [ ]:
import pandas as pd
import glob
import re

months = [
    'Januari',
    'Februari',
    'Maret',
    'April',
    'Mei',
    'Juni',
    'Juli',
    'Agustus',
    'September',
    'Oktober',
    'November',
    'Desember'
]
month_mapping = {
    'Januari': '01',
    'Februari': '02',
    'Maret': '03',
    'April': '04',
    'Mei': '05',
    'Juni': '06',
    'Juli': '07',
    'Agustus': '08',
    'September': '09',
    'Oktober': '10',
    'November': '11',
    'Desember': '12'
}

In [ ]:
#@title Load data inflasi Indonesia

inf_files = glob.glob("../Data/Train_data/Inflasi Bulanan*.csv")
list_inf_df = []

for file in inf_files:
    #regex mengambil tahun dari nama file
    tahun = re.search(r'\d{4}', file).group()

    df_inf = pd.read_csv(file, skiprows=3)

    #Filter untuk menggunakan data inflasi Nasional (INDONESIA)
    df_indo = df_inf[df_inf['Unnamed: 0'].astype(str).str.strip().str.upper() == 'INDONESIA'].copy()

    #ubah susunan data dari wide ke long
    df_melt = df_indo.melt(id_vars=['Unnamed: 0'], value_vars=months, var_name='Bulan', value_name='Inflasi')
    df_melt['Tahun'] = tahun
    list_inf_df.append(df_melt)

#merge seluruh data inflasi
inf_gabung = pd.concat(list_inf_df, ignore_index=True)
inf_gabung['Inflasi'] = pd.to_numeric(inf_gabung['Inflasi'], errors='coerce')

In [ ]:
#@title Load data jumlah penumpang kereta

pen_files = glob.glob("../Data/Train_data/Jumlah Penumpang*.csv")
list_pen_df = []

for file in pen_files:
    #regex mengambil tahun dari nama file
    tahun = re.search(r'\d{4}', file).group()

    df_pen = pd.read_csv(file, skiprows=3)
    df_pen.rename(columns={'Unnamed: 0': 'Kategori'}, inplace=True)

    #ubah susunan data dari wide ke long
    df_melt = df_pen.melt(id_vars=['Kategori'], value_vars=months, var_name='Bulan', value_name='Jumlah_Penumpang_Ribu')
    df_melt['Tahun'] = tahun
    list_pen_df.append(df_melt)

#merge seluruh data penumpang (2024, 2025)
pen_gabung = pd.concat(list_pen_df, ignore_index=True)
pen_gabung['Jumlah_Penumpang_Ribu'] = pd.to_numeric(pen_gabung['Jumlah_Penumpang_Ribu'], errors='coerce')

In [ ]:
#@title Merge data inflasi Indonesia dan jumlah penumpang kereta

#merge data data inflasi Indonesia dan jumlah penumpang kereta berdasarkan Tahun dan Bulan
df_final = pd.merge(pen_gabung, inf_gabung[['Tahun', 'Bulan', 'Inflasi']], on=['Tahun', 'Bulan'], how='left')

#membuat kolom tanggal (datetime)
df_final['Bulan_Angka'] = df_final['Bulan'].map(month_mapping)
df_final['Tanggal'] = pd.to_datetime(df_final['Tahun'] + '-' + df_final['Bulan_Angka'])

#merapikan susunan kolom
df_final = df_final[['Tanggal', 'Tahun', 'Bulan', 'Kategori', 'Jumlah_Penumpang_Ribu', 'Inflasi']]
df_final['Kategori'] = df_final['Kategori'].str.strip()

#hapus baris total dan jawa (jabodetabek + non jabodetabek) dari jumlah penumpang kereta
df_final = df_final[~df_final['Kategori'].isin(['Total', 'Jawa (Jabodetabek+Non Jabodetabek)'])]

#sort by tanggal
df_final.sort_values(by=['Tanggal', 'Kategori'], inplace=True)

#export to csv
df_final.to_csv("cleaned_data.csv", index=False)

In [ ]:
cek_df = pd.read_csv("cleaned_data.csv")
cek_df.shape
cek_df.info()

# Save cleaned data to aiven

In [ ]:
!pip install sqlalchemy pymysql

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text

In [ ]:
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine

load_dotenv()

host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

#inisiasi db
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")

In [ ]:
#create table and column
with engine.connect() as conn:
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS forecast_kereta (
    id INT AUTO_INCREMENT PRIMARY KEY,
    Tanggal DATE,
    Tahun TEXT,
    Bulan TEXT,
    Kategori VARCHAR(100),
    Jumlah_Penumpang_Ribu FLOAT,
    Inflasi FLOAT,
    UNIQUE (Tanggal, Kategori)
);
"""))

In [ ]:
#insert data ke DB (cukup sekali aja, nanti kalo ada perubahan data baru run lagi)
from sqlalchemy import text
# with engine.connect() as conn:
#     # Delete all existing data while preserving the table schema
#     conn.execute(text("DELETE FROM forecast_kereta;"))
#     conn.commit()
df_final.to_sql(
    name='forecast_kereta',
    con=engine,
    if_exists='append', # Changed to 'append' after deleting existing data
    index=False
)

In [ ]:
pd.read_sql(
    text("SHOW TABLES"),
    engine
)

In [ ]:
pd.read_sql(
    text("DESCRIBE forecast_kereta"),
    engine
)

In [ ]:
pd.read_sql(
    text("SELECT * FROM forecast_kereta WHERE Tahun = 2024 limit 15;"),
    engine
)